In [85]:
import numpy as np
import polars as pl
import statsmodels.api as sm
from sklearn.metrics import root_mean_squared_error

# Base setup

In [86]:
rng = np.random.default_rng(42)

N = 200000

x = rng.normal(0,1,size=N)
v = rng.normal(0,1,size=N)
u = rng.normal(0,1,size=N)

# 4 DGP
d_i = 1_(x_i * lambda_0 + v_i > 0)

In [87]:
# 1: with lambda_0 = 0
d1 = pl.DataFrame(v)
d1 = ((d1.with_columns((pl.when(pl.col("column_0") > 0).then(1).otherwise(0)).alias("d"))).drop(pl.col("column_0")))

# 2: with lambda_0 = 5
d2 = x * 5 + v
d2 = pl.DataFrame(d2)
d2 = ((d2.with_columns((pl.when(pl.col("column_0") > 0).then(1).otherwise(0)).alias("d"))).drop(pl.col("column_0")))

DGP : y_i = 1 + d_i * a_0 + (x_i * B_O)^l_0 + u_i

In [88]:
# 1: With a_0 = 1, B_0 = 1, l_0 = 1 -> y_i = 1 + d1_i + x_i + u_i
y1 = d1.with_columns(**{"col": pl.col("d") + 1, "u": u, "x": x}).with_columns((1+ pl.col("d") + pl.col("u") + pl.col("x")).alias("y")).drop("col")
# 1 + d2 + x^2 + u
# 1 + d1 + x + u

# 2: With # With a_0 = 1, B_0 = 1, l_0 = 2 -> y_i = 1 + d1_i + x_i^2 + u_i
y2 = d1.with_columns(**{"col": pl.col("d") + 1, "u": u, "x2": x**2, "x": x}).with_columns((1+ pl.col("d") + pl.col("u") + pl.col("x2")).alias("y")).drop("col")
# 1 + d2 + x^2 + u
# 1 + d1 + x^2 + u

# 3: With a_0 = 1, B_0 = 1, l_0 = 1 -> y_i = 1 + d2_i + x_i + u_i
y3 = d2.with_columns(**{"col": pl.col("d") + 1, "u": u, "x": x}).with_columns((1+ pl.col("d") + pl.col("u") + pl.col("x")).alias("y")).drop("col")
# 1 + d2 + x^2 + u
# 1 + d2 + x + u

# 4: With # With a_0 = 1, B_0 = 1, l_0 = 2 -> y_i = 1 + d2_i + x_i^2 + u_i
y4 = d2.with_columns(**{"col": pl.col("d") + 1, "u": u, "x2": x**2, "x": x}).with_columns((1+ pl.col("d") + pl.col("u") + pl.col("x2")).alias("y")).drop("col")
# 1 + d2 + x^2 + u

# Predictions

#### T and P datasets

In [89]:
y1t = y1.head(100000)
y1p = y1.tail(100000)
y2t = y2.head(100000)
y2p = y2.tail(100000)
y3t = y3.head(100000)
y3p = y3.tail(100000)
y4t = y4.head(100000)
y4p = y4.tail(100000)

In [90]:
X1 = y1t.select(["x", "d"]).to_numpy()
X1 = sm.add_constant(X1)
X1p = y1p.select(["x", "d"]).to_numpy()
X1p = sm.add_constant(X1p)

X2 = y2t.select(["x","d"]).to_numpy()
X2 = sm.add_constant(X2)
X2p = y2p.select(["x","d"]).to_numpy()
X2p = sm.add_constant(X2p)

X3 = y3t.select(["x", "d"]).to_numpy()
X3 = sm.add_constant(X3)
X3p = y3p.select(["x", "d"]).to_numpy()
X3p = sm.add_constant(X3p)

X4 = y4t.select(["x", "d"]).to_numpy()
X4 = sm.add_constant(X4)
X4p = y4p.select(["x", "d"]).to_numpy()
X4p = sm.add_constant(X4p)

Run the regression on T data

In [91]:
# on y1
mod1t = sm.OLS(y1t["y"].to_list(), X1).fit()

# on y2
mod2t = sm.OLS(y2t["y"].to_list(), X2).fit()

# on y3
mod3t = sm.OLS(y3t["y"].to_list(), X3).fit()

# on y4
mod4t = sm.OLS(y4t["y"].to_list(), X4).fit()

Run the predictions on T and P data

In [92]:
# on y1t 

predt1 = mod1t.predict(X1)
predp1 = mod1t.predict(X1p)

# on y2t

predt2 = mod2t.predict(X2)
predp2 = mod2t.predict(X2p)

# on y3t

predt3 = mod3t.predict(X3)
predp3 = mod3t.predict(X3p)

# on y4t

predt4 = mod4t.predict(X4)
predp4 = mod4t.predict(X4p)


Now compute root mean squared error

In [ ]:
# on y1
rmset1 = root_mean_squared_error(y1t["y"].to_numpy(), predt1)
rmsep1 = root_mean_squared_error(y1p["y"].to_numpy(), predp1)

# on y2
rmset2 = root_mean_squared_error(y2t["y"].to_numpy(), predt2)
rmsep2 = root_mean_squared_error(y2p["y"].to_numpy(), predp2)

# on y3
rmset3 = root_mean_squared_error(y3t["y"].to_numpy(), predt3)
rmsep3 = root_mean_squared_error(y3p["y"].to_numpy(), predp3)

# on y4
rmset4 = root_mean_squared_error(y4t["y"].to_numpy(), predt4)
rmsep4 = root_mean_squared_error(y4p["y"].to_numpy(), predp4)

print(f"RMSE on y1: {rmset1}, RMSE on y1p: {rmsep1}")
print(f"RMSE on y2: {rmset2}, RMSE on y2p: {rmsep2}")
print(f"RMSE on y3: {rmset3}, RMSE on y3p: {rmsep3}")
print(f"RMSE on y4: {rmset4}, RMSE on y4p: {rmsep4}")

RMSE on y1: 1.0015081942521897, RMSE on y1p: 0.9994641655544368
RMSE on y2: 1.749098430841376, RMSE on y2p: 1.7388904273738912
RMSE on y3: 1.0015036379782345, RMSE on y3p: 0.9994535677076822
RMSE on y4: 1.74910439793059, RMSE on y4p: 1.738896247960391


Now we add also the covariates ^2 to the model

In [94]:
# Create the new X matrices with x^2

X1s = sm.add_constant(y1t.select(pl.col(["x","d"])).with_columns(**{"x2": pl.col("x")**2}).to_numpy())
X1ps = sm.add_constant(y1p.select(pl.col(["x","d"])).with_columns(**{"x2": pl.col("x")**2}).to_numpy())
X2s = sm.add_constant(y2t.select(pl.col(["x","d"])).with_columns(**{"x2": pl.col("x")**2}).to_numpy())
X2ps = sm.add_constant(y2p.select(pl.col(["x","d"])).with_columns(**{"x2": pl.col("x")**2}).to_numpy())
X3s = sm.add_constant(y3t.select(pl.col(["x","d"])).with_columns(**{"x2": pl.col("x")**2}).to_numpy())
X3ps = sm.add_constant(y3p.select(pl.col(["x","d"])).with_columns(**{"x2": pl.col("x")**2}).to_numpy())
X4s = sm.add_constant(y4t.select(pl.col(["x","d"])).with_columns(**{"x2": pl.col("x")**2}).to_numpy())
X4ps = sm.add_constant(y4p.select(pl.col(["x","d"])).with_columns(**{"x2": pl.col("x")**2}).to_numpy())


run the regressions 

In [95]:
# on y1

modts1 = sm.OLS(y1t["y"].to_list(), X1s).fit()
predts1 = modts1.predict(X1s)
predps1 = modts1.predict(X1ps)

# on y2
modts2 = sm.OLS(y2t["y"].to_list(), X2s).fit()
predts2 = modts2.predict(X2s)
predps2 = modts2.predict(X2ps)

# on y3
modts3 = sm.OLS(y3t["y"].to_list(), X3s).fit()
predts3 = modts3.predict(X3s)
predps3 = modts3.predict(X3ps)

# on y4
modts4 = sm.OLS(y4t["y"].to_list(), X4s).fit()
predts4 = modts4.predict(X4s)
predps4 = modts4.predict(X4ps)

Now compute root mean squared error

In [96]:
# on y1
rmset1 = root_mean_squared_error(y1t["y"].to_numpy(), predts1)
rmsep1 = root_mean_squared_error(y1p["y"].to_numpy(), predps1)

# on y2
rmset2 = root_mean_squared_error(y2t["y"].to_numpy(), predts2)
rmsep2 = root_mean_squared_error(y2p["y"].to_numpy(), predps2)

# on y3
rmset3 = root_mean_squared_error(y3t["y"].to_numpy(), predts3)
rmsep3 = root_mean_squared_error(y3p["y"].to_numpy(), predps3)

# on y4
rmset4 = root_mean_squared_error(y4t["y"].to_numpy(), predts4)
rmsep4 = root_mean_squared_error(y4p["y"].to_numpy(), predps4)

print(f"RMSE on y1: {rmset1}, RMSE on y1p: {rmsep1}")
print(f"RMSE on y2: {rmset2}, RMSE on y2p: {rmsep2}")
print(f"RMSE on y3: {rmset3}, RMSE on y3p: {rmsep3}")
print(f"RMSE on y4: {rmset4}, RMSE on y4p: {rmsep4}")

RMSE on y1: 1.0015074623784335, RMSE on y1p: 0.9994674065033649
RMSE on y2: 1.0015074623784335, RMSE on y2p: 0.9994674065033649
RMSE on y3: 1.001502900894516, RMSE on y3p: 0.9994568363532069
RMSE on y4: 1.001502900894516, RMSE on y4p: 0.9994568363532069


Which give the best: When the DGP includes quadratic, the model with X^2 is substantially better. When the model doesnt, adding X^2 doesnt change much RMSE.

# Causal Analysis

For all 4 DGPs the true treatment effect is alpha_0 = 1 or given by D in our analysis

In [ ]:
models_linear = [mod1t, mod2t, mod3t, mod4t]
models_squared = [modts1, modts2, modts3, modts4]

print(modts1.summary())

for i in range(4):
    alpha_linear = models_linear[i].params[2]
    alpha_squared = models_squared[i].params[2]

    print(f"DGP {i+1}")
    print(f"  Linear:    Effect = {alpha_linear:.4f}, Error = {alpha_linear - 1:.4f}")
    print(f"  Quadratic: Effect = {alpha_squared:.4f}, Error = {alpha_squared - 1:.4f}")

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.557
Model:                            OLS   Adj. R-squared:                  0.557
Method:                 Least Squares   F-statistic:                 6.279e+04
Date:                Wed, 11 Nov 2026   Prob (F-statistic):               0.00
Time:                        11:41:49   Log-Likelihood:            -1.4204e+05
No. Observations:              100000   AIC:                         2.841e+05
Df Residuals:                   99997   BIC:                         2.841e+05
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.9964      0.004    222.370      0.0